In [ ]:
from datetime import datetime
import math
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pytz

In [ ]:
# Load the dataset
apps_df = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')
apps_df.info() 

In [ ]:
# Drop missing values
apps_df = apps_df.drop_duplicates().copy()
apps_df = apps_df.dropna(subset=['App', 'Category', 'Reviews', 'Installs', 'Last Updated']).copy()

# Clean Installs in-place
apps_df['Installs'] = apps_df['Installs'].astype(str).str.replace(',', '', regex=False).str.replace('+', '', regex=False)

apps_df = apps_df[apps_df['Installs'].str.isnumeric()].copy()
apps_df['Installs'] = apps_df['Installs'].astype(int)

# Clean Reviews in-place
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')

# Convert Last Updated to datetime in-place
apps_df['Last Updated'] = pd.to_datetime(
    apps_df['Last Updated'], errors='coerce'
)
apps_df.info()

In [ ]:
# Get all unique categories from the dataset
unique_categories = apps_df['Category'].dropna().unique()

# Filter for categories starting with B, C, or E
task4_categories = []
for category in unique_categories:
  category_str = str(category).upper()
  if (
      category_str.startswith('B')
      or category_str.startswith('C')
      or category_str.startswith('E')
      or category_str == 'DATING'
  ):
    task4_categories.append(category)

# Sort alphabetically
task4_categories.sort()

# Print each category on its own line
print('--- Categories Starting with B, C, or E, plus DATING ---')
for cat in task4_categories:
  print(cat)

In [ ]:
# Function to check app name criteria
def is_valid_app_name(name):
  name_str = str(name).strip()

  # Check if starts with X, Y, or Z (case-insensitive)
  if name_str.upper().startswith(('X', 'Y', 'Z')):
    return False

  # Check if contains capital 'S' (strictly case-sensitive)
  if 'S' in name_str:
    return False

  return True

In [ ]:
# Translation dictionary
category_translation_map = {
    'BEAUTY': 'सुंदरता',  # Hindi
    'BUSINESS': 'வணிகம்',  # Tamil
    'DATING': 'Partnersuche',  # German
}

# Filter conditions
cond_reviews = apps_df['Reviews'] > 500
cond_category = apps_df['Category'].isin(task4_categories)
cond_app_name = apps_df['App'].apply(is_valid_app_name)

# Apply filters
task4_filtered_df = apps_df[
    cond_reviews & cond_category & cond_app_name
].copy()

# Function to get display category name
def get_display_category(category):
  return category_translation_map.get(category, category)

# Apply localized names for display
task4_filtered_df['Category_Display'] = task4_filtered_df['Category'].apply(
    get_display_category
)

print(f'Remaining rows: {len(task4_filtered_df)}')
print(task4_filtered_df['Category_Display'].value_counts())

In [ ]:
# Extract Year-Month (Period) from Last Updated
task4_filtered_df['Year_Month'] = task4_filtered_df[
    'Last Updated'
].dt.to_period('M')

# Group by Year_Month and Category_Display to aggregate Total Installs
monthly_installs = (
    task4_filtered_df.groupby(['Year_Month', 'Category_Display'])['Installs']
    .sum()
    .reset_index()
)

# Convert Year_Month back to datetime format for smooth Plotly time-series plotting
monthly_installs['Year_Month_DT'] = (
    monthly_installs['Year_Month'].dt.to_timestamp()
)

# Sort strictly by Category and Date before calculating MoM growth
monthly_installs = monthly_installs.sort_values(
    ['Category_Display', 'Year_Month_DT']
)

# Calculate Month-over-Month (MoM) Percentage Growth per category
monthly_installs['MoM_Growth'] = monthly_installs.groupby('Category_Display')[
    'Installs'
].pct_change()

# Flag periods where growth strictly exceeds 20% (> 0.20)
monthly_installs['Exceeds_20pct'] = monthly_installs['MoM_Growth'] > 0.20

# Inspect aggregated results
monthly_installs.head(10)

In [ ]:
# Check time (6 PM - 9 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime('18:00:00', '%H:%M:%S').time()
end_time = datetime.strptime('21:00:00', '%H:%M:%S').time()

is_within_time = start_time <= current_time < end_time

# Prepare HTML summary table rows
category_summary = (
    monthly_installs.groupby('Category_Display')['Installs']
    .sum()
    .reset_index()
)
cat_rows_html = ''
for idx, row in category_summary.iterrows():
  cat_rows_html += f"<tr><td>{row['Category_Display']}</td><td>{int(row['Installs']):,}</td></tr>\n"

# Defined file paths
graph_path = 'task_4_graph.html'
dashboard_path = 'task_4_dashboard.html'

In [ ]:
# Generating the graph using Plotly and saving it as task_4_graph.html silently if within the allowed time window
if is_within_time:
  categories = sorted(monthly_installs['Category_Display'].dropna().unique())

  rows, cols = 3, 3
  categories_per_page = rows * cols
  number_of_pages = math.ceil(len(categories) / categories_per_page)

  # Collect HTML snippets for each page figure
  fig_html_blocks = []

  for page in range(number_of_pages):
    start_idx = page * categories_per_page
    end_idx = start_idx + categories_per_page
    page_categories = categories[start_idx:end_idx]

    subplot_titles = [str(cat) for cat in page_categories]
    while len(subplot_titles) < 9:
      subplot_titles.append('')

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.12,
        horizontal_spacing=0.07,
    )

    for index, cat in enumerate(page_categories):
      row = (index // cols) + 1
      col = (index % cols) + 1

      cat_df = monthly_installs[
          monthly_installs['Category_Display'] == cat
      ].sort_values('Year_Month_DT')

      y_baseline = 0

      # Add >20% MoM growth shaded polygons
      for i in range(1, len(cat_df)):
        prev_row = cat_df.iloc[i - 1]
        curr_row = cat_df.iloc[i]

        if curr_row['MoM_Growth'] > 0.20:
          fig.add_trace(
              go.Scatter(
                  x=[
                      prev_row['Year_Month_DT'],
                      curr_row['Year_Month_DT'],
                      curr_row['Year_Month_DT'],
                      prev_row['Year_Month_DT'],
                  ],
                  y=[
                      y_baseline,
                      y_baseline,
                      curr_row['Installs'],
                      prev_row['Installs'],
                  ],
                  fill='toself',
                  fillcolor='rgba(255, 165, 0, 0.25)',
                  line=dict(width=0),
                  hoverinfo='skip',
                  showlegend=False,
              ),
              row=row,
              col=col,
          )

      # Add line trace
      fig.add_trace(
          go.Scatter(
              x=cat_df['Year_Month_DT'],
              y=cat_df['Installs'],
              mode='lines',
              name=cat,
              line=dict(width=2),
              marker=dict(size=4),
              hovertemplate=(
                  '<b>%{fullData.name}</b><br>'
                  'Date: %{x|%b %Y}<br>'
                  'Installs: %{y:,.0f}<br>'
                  '<extra></extra>'
              ),
              showlegend=False,
          ),
          row=row,
          col=col,
      )

      fig.update_yaxes(
          tickformat='.2s',
          showgrid=True,
          title_text='Installs',
          row=row,
          col=col,
      )

      fig.update_xaxes(
          showgrid=True, tickformat='%b\n%Y', row=row, col=col
      )

    page_text = (
        f' — Page {page + 1} of {number_of_pages}' if number_of_pages > 1 else ''
    )
    fig.update_layout(
        title=dict(
            text=(
                'Monthly Total Installs by App Category'
                f'{page_text}<br>'
                '<sup>Shaded polygons represent >20% month-over-month growth'
                ' periods</sup>'
            ),
            x=0.5,
            xanchor='center',
        ),
        height=900,
        template='plotly_white',
        hovermode='closest',
        margin=dict(l=60, r=40, t=100, b=60),
    )

    fig_html_blocks.append(
        fig.to_html(full_html=False, include_plotlyjs='cdn')
    )

  # Write combined multi-page HTML graph file
  combined_graph_html = f"""<!DOCTYPE html>
<html>
<head>
    <title>Task 4 Graph</title>
</head>
<body style="background-color: #f4f6f9; padding: 20px;">
    {'<br><hr><br>'.join(fig_html_blocks)}
</body>
</html>"""

  with open(graph_path, 'w', encoding='utf-8') as f:
    f.write(combined_graph_html)

  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: green; font-weight: bold;">Status: Graph Successfully Generated</p>
        <p>The interactive multi-panel time-series graph has been generated and saved to <b>{graph_path}</b> because the execution occurred within the allowed window (6:00 PM - 9:00 PM IST).</p>
        <p><a href="{graph_path}" target="_blank" style="display: inline-block; padding: 10px 15px; background-color: #1a73e8; color: white; text-decoration: none; border-radius: 5px;">Open Graph</a></p>
    </div>
  """

else:
  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: red; font-weight: bold;">Status: Graph Restricted</p>
        <p>The interactive graph is only generated and accessible between <b>6:00 PM IST and 9:00 PM IST</b>. The graph file ({graph_path}) was not created. Please rerun the code during the allowed time window.</p>
    </div>
  """

In [ ]:
# Generating the HTML dashboard content
dashboard_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Play Store Installs Dashboard - Task 4</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 30px;
            background-color: #f4f6f9;
            color: #333;
        }}
        h1 {{ color: #1a73e8; }}
        .card {{
            background: white;
            padding: 20px;
            margin-bottom: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid #ddd;
            text-align: left;
        }}
        th {{ background-color: #f0f4f8; }}
    </style>
</head>
<body>

    <h1>Play Store App Analysis Dashboard (Task 4)</h1>

    <div class="card">
        <h2>Dashboard Summary</h2>
        <p><b>Filter Rules Applied:</b> Reviews &gt; 500, Categories starting with B, C, E, or DATING, App Name excludes 'X', 'Y', 'Z' prefixes and capital 'S'.</p>
        <p><b>Total Filtered Rows Analyzed:</b> {len(task4_filtered_df):,}</p>
    </div>

    <div class="card">
        <h2>Qualifying Categories (Total Historical Installs)</h2>
        <table>
            <tr>
                <th>Category Name (Localized)</th>
                <th>Total Combined Installs</th>
            </tr>
            {cat_rows_html}
        </table>
    </div>

    {status_section_html}

</body>
</html>
"""

with open(dashboard_path, 'w', encoding='utf-8') as f:
  f.write(dashboard_html_content)